# Lab 2: Business Text Data

**Business question:** Can our customer messages support the follow-up decision, and what counts as a correct label?

This lab has three parts:

| Part | Data | Time | What you hand in |
|---|---|---|---|
| 1 Guided core | 24 customer messages | about 75–90 min | Labels, short answers, saved files and the checkpoint |
| 2 Practice | CFPB complaint extract | about 75–100 min | Three exercises and the interpretation |
| 3 Challenge | Both | optional | Nothing required |

This is the longest lab so far: plan on about 3 hours in total, and start early in the week. Most of the time goes to reading and writing, not to code.

Graded with the course Lab Notebook rubric in Canvas.

You need the Module 1 setup. No new package or outside service is used.

## Part 1 Guided core

In Part 1 you check the 24 messages from Module 1, label 12 of them with our review policy and look at the data split we supply. Short code cells do the counting. Your written answers show your judgment.

### 1.1 Load the tools

Select the `ban6403` kernel in VS Code, then run the cell below. No output means it worked. If the import fails, check that `ban6403` is selected and that pandas is installed in it. The Module 0 readiness notebook shows what to fix.

In [ ]:
# Path builds file paths; pandas reads and inspects tables.
from pathlib import Path
import pandas as pd

### 1.2 Point to the data folder

As in Lab 1, the notebook runs from the `notebooks` folder, so the data folder is one level up: `../data`. We store that path once as `DATA`. This cell shows no output.

In [ ]:
# ".." means one folder up: the data folder next to notebooks/.
DATA = Path("../data")

### 1.3 Read the customer messages

This is the same file you used in Module 1. `keep_default_na=False` keeps an empty message as an empty string, so we can find it later. The original file is never changed.

You should see the first three rows: FB001, FB002 and FB003.

In [ ]:
# keep_default_na=False keeps an empty message as "" instead of NaN.
feedback = pd.read_csv(DATA / "raw/customer_feedback_m01.csv", keep_default_na=False)
feedback.head(3)

### 1.4 Count messages by channel

The **unit of analysis** is what one row stands for. Here one row is one feedback message, not one customer. `value_counts()` counts the values in one column.

Expect 8 reviews, 8 emails and 8 chats. We chose these equal counts for the course. They do not show how real customers contact the business.

In [ ]:
# Count the messages in each channel.
feedback["channel"].value_counts()

In one or two sentences: what would you need to know about real customer contacts before you make a claim about the business? Think of dates, channel volumes or which customers wrote.

**Your answer:**

### 1.5 Find missing text

`str.strip()` removes spaces at the start and end, only for this check. The raw text stays unchanged.

Expect one row: FB022, a 3-star tablet review with no text. Read the row before you decide what to do with it.

In [ ]:
# .eq("") compares each value with the empty string: True for each message that is empty after removing outer spaces.
blank_text = feedback["text"].str.strip().eq("")
# .loc[rows] keeps the rows where the condition is True.
feedback.loc[blank_text]

In one or two sentences: what evidence is missing in FB022, and why can its 3-star rating not stand in for the text?

**Your answer:**

### 1.6 Find repeated text

`duplicated(keep=False)` marks every row whose text also appears in another row. We leave out the empty message, so missing text is not counted as a repeat.

Expect two rows: FB005 (chat) and FB020 (email), both with "The refund arrived yesterday. No action is required."

In [ ]:
# True for every row whose text also appears in another row.
same_text = feedback["text"].duplicated(keep=False)

# & means "and", ~ means "not": repeated text that is not the empty message.
repeated_text = same_text & ~blank_text
# .loc[rows, columns]: the rows where repeated_text is True, and three columns.
feedback.loc[repeated_text, ["feedback_id", "channel", "text"]]

In one or two sentences: do we know that FB005 and FB020 come from the same customer? What would change if you deleted one of them right away?

**Your answer:**

### 1.7 Read the review policy and prepare a worksheet

Read [the draft review policy](../docs/m02_annotation_guide.md). Version 1 (v1) has three labels:

- `review`: an open problem or a clear request for help. A question that asks how to do something counts as a request for help.
- `no_review`: the problem is clearly solved, or the customer gives an opinion and needs no help.
- `unclear`: too little context to decide. Staff would ask the customer for more details.

A label says whether staff should look at a message. It does not approve a refund or any other action. Use only the text you can see. A star rating alone does not decide the label.

The cell below makes a worksheet for FB001–FB012. `.set_index()` lets us pick rows by message ID. The `label` and `evidence` columns start empty, so unfinished rows stay easy to see. We do not label the protected messages.

In [ ]:
# Use the message IDs as row labels, so we can pick rows by ID.
messages = feedback.set_index("feedback_id")

# "FB001":"FB012" selects every row from FB001 to FB012, both included.
# .copy() makes a separate table, so adding columns does not touch messages.
labels = messages.loc["FB001":"FB012", ["text"]].copy()

# Both columns start empty. You fill them in step 1.8.
labels["label"] = ""
labels["evidence"] = ""

### 1.8 Label 12 messages

Each cell below prints one message. Then it stores your label and evidence for that message.

To label a message, type inside the two empty quotes and rerun the cell. For example:

`labels.loc["FB008", ["label", "evidence"]] = ["review", "Please check the bill"]`

- The label must be `review`, `no_review` or `unclear`, in lower case.
- The evidence is a short phrase copied exactly from the message.
- Label on your own first. Compare with a partner afterwards, in step 1.11.
- FB003 is done for you as an example.

Your edits stay in the cells when you save the notebook, so Run All keeps your answers.

#### FB001

The rating is five stars. What does the missing charger add?

In [ ]:
print(messages.loc["FB001", "text"])
# TODO: enter your label and evidence inside the quotes, then rerun this cell.
labels.loc["FB001", ["label", "evidence"]] = ["", ""]

#### FB002

Read the whole message. Is the problem still open, or already solved?

In [ ]:
print(messages.loc["FB002", "text"])
# TODO: enter your label and evidence inside the quotes, then rerun this cell.
labels.loc["FB002", ["label", "evidence"]] = ["", ""]

#### FB003 (example, done for you)

The customer says the headset stopped working and asks for a refund. That is an open problem and a clear request, so the label is `review`. The evidence quotes the request. The label means staff should look at the message; it does not approve the refund.

In [ ]:
print(messages.loc["FB003", "text"])
# Example: a clear request for help needs staff review.
labels.loc["FB003", ["label", "evidence"]] = ['review', 'Please arrange a refund']

#### FB004

Does this opinion show a need for help?

In [ ]:
print(messages.loc["FB004", "text"])
# TODO: enter your label and evidence inside the quotes, then rerun this cell.
labels.loc["FB004", ["label", "evidence"]] = ["", ""]

#### FB005

The word *refund* appears. Is there still a request?

In [ ]:
print(messages.loc["FB005", "text"])
# TODO: enter your label and evidence inside the quotes, then rerun this cell.
labels.loc["FB005", ["label", "evidence"]] = ["", ""]

#### FB006

Which phrase shows a request for help?

In [ ]:
print(messages.loc["FB006", "text"])
# TODO: enter your label and evidence inside the quotes, then rerun this cell.
labels.loc["FB006", ["label", "evidence"]] = ["", ""]

#### FB007

Which phrase describes how the purchase went?

In [ ]:
print(messages.loc["FB007", "text"])
# TODO: enter your label and evidence inside the quotes, then rerun this cell.
labels.loc["FB007", ["label", "evidence"]] = ["", ""]

#### FB008

What does the customer ask staff to check?

In [ ]:
print(messages.loc["FB008", "text"])
# TODO: enter your label and evidence inside the quotes, then rerun this cell.
labels.loc["FB008", ["label", "evidence"]] = ["", ""]

#### FB009

The customer asks how to do something. Check the `review` line of the policy in 1.7: how does it treat a how-to question?

In [ ]:
print(messages.loc["FB009", "text"])
# TODO: enter your label and evidence inside the quotes, then rerun this cell.
labels.loc["FB009", ["label", "evidence"]] = ["", ""]

#### FB010

Does a different color show a current need for help under v1? If you are unsure, use this message in your note in step 1.11.

In [ ]:
print(messages.loc["FB010", "text"])
# TODO: enter your label and evidence inside the quotes, then rerun this cell.
labels.loc["FB010", ["label", "evidence"]] = ["", ""]

#### FB011

The message mentions a replacement. Is anything still wrong now?

In [ ]:
print(messages.loc["FB011", "text"])
# TODO: enter your label and evidence inside the quotes, then rerun this cell.
labels.loc["FB011", ["label", "evidence"]] = ["", ""]

#### FB012

What context is missing? Use the policy; do not invent a history for the customer.

In [ ]:
print(messages.loc["FB012", "text"])
# TODO: enter your label and evidence inside the quotes, then rerun this cell.
labels.loc["FB012", ["label", "evidence"]] = ["", ""]

### 1.9 Review your worksheet

This shows all 12 rows. Check that each label follows the policy and that each evidence phrase supports it. To change an answer, edit that message's cell in 1.8 and rerun it.

In [ ]:
# Show the whole worksheet.
labels

### 1.10 Find unfinished labels

This counts rows where the label or the evidence is still empty and lists their IDs.

If you have not labeled anything yet, expect `Messages still to label: 11 of 12` (only the FB003 example is filled). When you finish, expect `0 of 12`. Zero only means every row has an entry. It does not mean the labels are right.

In [ ]:
# True when the label is empty, or (|) when the evidence is empty.
no_label = labels["label"].str.strip().eq("")
no_evidence = labels["evidence"].str.strip().eq("")
unfinished = no_label | no_evidence

# sum() counts the True values; labels.index holds the message IDs.
print("Messages still to label:", unfinished.sum(), "of", len(labels))
print("IDs still to label:", list(labels.index[unfinished]))

### 1.11 Decide one disagreement

People can read the same message differently. Deciding a disagreement and recording the reason is called **adjudication**. The written record is an adjudication note.

Pick one hard message (FB010 is a good choice). Compare your label with a partner's. If you work alone, write your own first label and one other label a reasonable person might choose.

Keep both first labels in the note. If your final label changes, also update that message's cell in 1.8. If the policy needs a new rule, write it and give it a new version number, such as v1.1. If you adopt revised wording, recheck all 12 development messages under that version. If the wording is only a proposal, keep submitted labels under v1.

**Your answer:**

- ID:
- My first label and evidence:
- Partner's label (or my own alternative) and reason:
- Final label and reason:
- Suggested policy wording and version:

### 1.12 Read the partition file

A **partition** is a set of records with a fixed role. The partition file gives every message ID a group, a partition and a reason.

Expect development 13, protected 10 and excluded 1. The 12 messages you labeled are part of development. The 13th development record is FB020, the repeat of FB005.

In [ ]:
# Read the partition file and count the records in each partition.
partition = pd.read_csv(DATA / "partitions/m02_partition.csv")
partition["partition"].value_counts()

### 1.13 List the protected IDs

Expect 10 IDs: FB013–FB019, FB021, FB023 and FB024. These messages are kept for a fair test in later modules.

You may look at the IDs. Do not label these messages, and do not use them to adjust your policy or rules. To **tune** means to adjust a method until it scores better. Tuning on the test records would make the test unfair.

In [ ]:
# Keep the rows of the protected partition, and only the ID column.
is_protected = partition["partition"] == "protected"
protected_ids = partition.loc[is_protected, ["feedback_id"]]
protected_ids

### 1.14 Check the repeated-text group

Expect two rows: FB005 and FB020, both in group `exact_refund_text` and both in development. Keeping them together means the same wording cannot appear in both the build set and the test set later. The group does not prove that one customer wrote both messages.

In [ ]:
# The rows of the partition file that belong to the repeated-text group.
partition.loc[partition["group_id"] == "exact_refund_text"]

In two or three sentences: why do FB005 and FB020 stay in the same partition? Name one stronger ID you would want for grouping in a real project. Why are 10 protected made-up messages too few to measure accuracy?

**Your answer:**

### 1.15 Write a short corpus card

A **corpus** is a collection of text records chosen for one purpose. A corpus card is a short description that lets another analyst understand it. Use what you found above, not a general list of risks. One line per item is enough.

**Your answer:**

- Decision and owner:
- Unit, source and size:
- Fields, and context we do not have:
- Missing and repeated text, and how we handle them:
- One coverage limit and what it means for our claims:
- Permitted use and where the data came from:
- Label policy, version and one hard boundary:
- Protected test set:

### 1.16 Save your outputs

These three cells write two small CSV files to `outputs/m02`. The raw data and the partition file are not changed.

- `review_labels.csv`: your 12 labels and evidence phrases, with the message ID. Any message you have not labeled stays empty in the file.
- `protected_ids.csv`: the 10 protected IDs, unchanged.

Expect no error, and the folder path printed at the end.

In [ ]:
OUTPUT_DIR = Path("../outputs/m02")
# parents=True also creates outputs/ if needed; exist_ok=True accepts an existing folder.
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
# The row labels are the message IDs; index_label names that column in the file.
labels.to_csv(OUTPUT_DIR / "review_labels.csv", index_label="feedback_id")

In [ ]:
# index=False leaves out pandas' row numbers; the file keeps only the ID column.
protected_ids.to_csv(OUTPUT_DIR / "protected_ids.csv", index=False)
print(OUTPUT_DIR)

### Checkpoint

Answer in the cell below (2–3 sentences in total).

1. Choose FB002 or FB010. Quote your evidence phrase and explain why the star rating does not decide the label.
2. Why must the 10 protected messages stay unlabeled while you write and adjust the policy?

**Your answer:**

## Part 2 Practice: real complaints

Now run the same kind of data check on real complaints that consumers sent to the U.S. Consumer Financial Protection Bureau (CFPB). The course extract has 10,000 complaints from July to December 2025. Each has the consumer's own words, the product they chose and how the company closed the complaint. It is already split into `train`, `valid` and `test`. Use `train` to build methods, `valid` to compare development choices, and `test` for final evaluation. In this lab, read and label only `train` narratives. Do not change the split.

**Where the data came from.** CFPB stopped publishing new complaint narratives in August 2026. This extract comes from CFPB's official narrative archive (see `data/course/README.md`). This record of where data came from is called **provenance**. A source can change or stop, so write down where and when you got your data.

### 2.1 Load the complaint extract

Expect `(10000, 13)`: 10,000 rows and 13 columns.

In [ ]:
# Read the complaints; shape gives (rows, columns).
complaints = pd.read_csv(DATA / "course/cfpb_complaints.csv")
complaints.shape

### 2.2 Check the unit of analysis

One row is one complaint. We check that each `complaint_id` appears only once, and we look at the date range.

Expect `True`, then the first and last dates in the extract.

In [ ]:
# is_unique is True when no complaint_id appears twice.
print("each complaint_id is unique:", complaints["complaint_id"].is_unique)
# The dates are written as YYYY-MM-DD, so min and max give the first and last day.
print("first date:", complaints["date_received"].min())
print("last date:", complaints["date_received"].max())

### 2.3 Count rows per product and split

`pd.crosstab` counts rows for each pair of values. `margins=True` adds the totals.

We chose how many complaints to keep for each product. So these counts describe the extract's design, not how common each problem is.

In [ ]:
# Rows: products. Columns: splits. margins=True adds an "All" row and column.
pd.crosstab(complaints["product"], complaints["split"], margins=True)

### 2.4 Count missing values in each column

`isna().sum()` counts empty cells in each column. When pandas reads this file, an empty cell becomes a missing value (`NaN`), not an empty string as in 1.3. So we also check for narratives that contain only spaces.

Look for the columns that have gaps, and check that `narrative` has none.

In [ ]:
# Number of missing values (NaN) in each column.
complaints.isna().sum()

In [ ]:
# Narratives that are present but contain only spaces.
complaints["narrative"].str.strip().eq("").sum()

### 2.5 Summarize narrative length by product

We count words by splitting each narrative on spaces. The new column `n_words` is added to our table in memory; the CSV file does not change.

Compare the medians across products. Also look at the smallest and largest values. The data README says we kept only narratives of 30 to 600 words. That is a coverage choice: very short and very long complaints are not in this extract.

In [ ]:
# split() turns each narrative into a list of words; str.len() counts them.
complaints["n_words"] = complaints["narrative"].str.split().str.len()

# One row per product, with four summaries of the word counts.
length_summary = complaints.groupby("product")["n_words"].agg(["min", "median", "mean", "max"])
length_summary.round(1)

### 2.6 Look for repeated text

We make a simple normalized copy of each narrative: lower case, with runs of spaces and line breaks turned into one space. Then we count narratives whose normalized text matches another one exactly.

The data README says many complaints were copies of one template letter. No two narratives in the extract are exactly the same, so expect 0.

In [ ]:
# Lower case, then split on any whitespace and join with single spaces.
lowered = complaints["narrative"].str.lower()
complaints["narrative_norm"] = lowered.str.split().str.join(" ")

# Count narratives whose normalized text also appears in another row.
complaints["narrative_norm"].duplicated(keep=False).sum()

### 2.7 Look for near-repeats

Template letters can still differ slightly. So we compare only the start of each narrative, its **opening**. We use the same rule as for the split: lower case, every run of non-letters (digits, punctuation, spaces) turned into one space, then the first 100 characters. `[^a-z]+` is a pattern that means "one or more characters that are not the letters a to z".

Complaints with the same opening are near-repeats worth reading.

In [ ]:
# Build the opening of each narrative in four small steps.
lowered = complaints["narrative"].str.lower()
# Replace every run of non-letters with one space.
letters_only = lowered.str.replace(r"[^a-z]+", " ", regex=True)
# Remove the spaces at both ends, then keep the first 100 characters.
complaints["opening"] = letters_only.str.strip().str[:100]

# True for every complaint whose opening also appears in another complaint.
shared = complaints["opening"].duplicated(keep=False)
print("complaints that share an opening:", shared.sum())
print("number of shared openings:", complaints.loc[shared, "opening"].nunique())

### 2.8 Check that the split keeps similar letters together

Remember FB005 and FB020. If a letter sits in `train` and a near-copy sits in `test`, a model can score well on the test just by remembering the template. The test would then look better than the model's work on truly new complaints. This is **leakage**.

To reduce this risk, we gave complaints with the same product and the same opening the same split. This is one grouping rule: it flags possible template matches. It can miss related complaints and group unrelated ones. You do not have to trust that we applied it. This cell checks it: for each product and opening, it counts the splits used. Expect every group to show 1.

In [ ]:
# For each product-and-opening group, count how many different splits it uses.
shared_complaints = complaints.loc[shared]
splits_per_group = shared_complaints.groupby(["product", "opening"])["split"].nunique()

# How many groups use 1 split, 2 splits, and so on.
splits_per_group.value_counts()

Now ignore the product and check each opening alone. A few openings appear under two or three products, and those can sit in different splits. We balanced the split within each product, so we formed the groups inside each product. This cell lists the openings that use more than one split. The `splits` column counts the distinct splits per opening. Expect three openings, with 23 complaints in total.

In [ ]:
# One row per opening. Each new column is written as name=(column, function):
# "size" counts the complaints; "nunique" counts the distinct splits and the distinct products.
opening_table = shared_complaints.groupby("opening").agg(
    complaints=("split", "size"),
    splits=("split", "nunique"),
    products=("product", "nunique"),
)

# Keep the openings that use more than one split.
opening_table.loc[opening_table["splits"] > 1]

Three shared openings cross splits, involving 23 complaints overall. This is a remaining leakage risk. These counts alone do not tell us how much it affects evaluation scores. Within each product, the check in 2.8 found no crossing at all. The data README lists this as a known limit.

Now look at the largest group. Count its complaints by product and split. Expect one product and one split.

In [ ]:
# value_counts() sorts from most to least common, so index[0] is the largest group.
largest = complaints.loc[shared, "opening"].value_counts().index[0]
print(largest)

# Keep the complaints with that opening and count them by product and split.
largest_group = complaints[complaints["opening"].eq(largest)]
pd.crosstab(largest_group["product"], largest_group["split"])

Counting is not enough: read the texts too. These two `train` complaints share an opening. Read both in full. Are they the same letter, or one template used for different problems? What is the same, and what differs?

This rule flags possible template matches. It can miss related complaints and group unrelated ones.

In [ ]:
# Two train complaints with the same opening, chosen by ID.
pair_ids = [16024833, 16026131]

# Use complaint_id as the row label, so we can look rows up by ID.
complaints_by_id = complaints.set_index("complaint_id")

# Print the ID, the split and the full narrative of each complaint.
for complaint_id in pair_ids:
    print(complaint_id, "|", complaints_by_id.loc[complaint_id, "split"])
    print(complaints_by_id.loc[complaint_id, "narrative"])
    print()

In one or two sentences: are 16024833 and 16026131 the same letter? What does reading them show that the count of shared openings does not?

**Your answer:**

We do not change the split in this course. Later modules use it as given.

### 2.9 Count redaction marks

CFPB replaces names and other personal details with `XXXX`. Dates are hidden as `XX/XX/XXXX` or as `XX/XX/year>`; both forms are common. Money amounts stay, written in braces such as `{$450.00}`. `regex=False` tells pandas to search for the exact characters.

In [ ]:
# True for each narrative that contains the exact characters.
has_xxxx = complaints["narrative"].str.contains("XXXX", regex=False)
has_amount = complaints["narrative"].str.contains("{$", regex=False)

# The mean of True/False values is a share (True counts as 1).
print("share with XXXX:", round(has_xxxx.mean(), 3))
print("share with an amount {$:", round(has_amount.mean(), 3))

### 2.10 Apply a label policy to real complaints

Every row here is a complaint, so almost every one would need review. We use a different business label instead. Suppose a payments team wants to know which complaints ask for money back.

Draft policy for `asks_for_money`:

- `yes`: the consumer clearly wants money returned or credited (refund, reimbursement, a reversed charge, a waived fee or released funds).
- `no`: the consumer wants something else (fix a report, stop calls, an explanation), and no money is sought.
- `unclear`: money is mentioned or lost, but the text does not show what the consumer wants.

As in Part 1, keep the ID and quote a short evidence phrase. We label only `train` complaints, so `valid` and `test` stay untouched. The cell below keeps them in a table named `train`; use it in the exercises. Read the three example complaints first.

In [ ]:
# The training complaints only; read and label only these.
train = complaints[complaints["split"].eq("train")]

example_ids = [16417628, 14384643, 14597922]

# Use complaint_id as the row label, then look up the three narratives by ID.
complaints_by_id = complaints.set_index("complaint_id")
example_texts = complaints_by_id.loc[example_ids, "narrative"]

# tolist() prints each narrative in full instead of cutting it short.
example_texts.tolist()

Here are labels for the three examples. The third one says money is missing but does not say what the consumer wants, so it is `unclear`.

In [ ]:
# One dictionary per labeled complaint; each key becomes a column.
practice_labels = pd.DataFrame([
    {"complaint_id": 16417628, "asks_for_money": "yes", "evidence": "I am requesting a refund"},
    {"complaint_id": 14384643, "asks_for_money": "no", "evidence": "Please remove this account from my report"},
    {"complaint_id": 14597922, "asks_for_money": "unclear", "evidence": "missing from my account"},
])
practice_labels

### Exercise 1: monetary relief by product

The column `company_response` records how the company closed each complaint. Compute the share of complaints with `company_response` equal to `"Closed with monetary relief"` for each product, and sort from highest to lowest.

Hint: `.eq(...)` gives True/False values, and the mean of True/False values is a share.

In [ ]:
# Exercise 1: your code here
# TODO: make a True/False column: company_response equals "Closed with monetary relief".
# TODO: group it by complaints["product"] and take the mean (the share per product).
# TODO: sort from highest to lowest with sort_values(ascending=False).

### Exercise 2: short narratives

1. Find the complaints with fewer than 40 words (use `n_words`). How many are there, and how many per product?
2. Show the `complaint_id` and full narrative of two short `train` complaints, and read them.

In [ ]:
# Exercise 2: your code here
# TODO: keep the complaints with n_words < 40 and print how many there are.
# TODO: count them per product with value_counts().
# TODO: show complaint_id and narrative of two short train complaints (tolist() shows the full text).

### Exercise 3: label five complaints

1. Choose five `train` complaints that are not the three examples in 2.10. You can use the short ones from Exercise 2 or search the `train` table from 2.10 for a word, for example `train["narrative"].str.contains("refund", case=False)`. If a word finds no complaints, choose another word.
2. Read each full narrative.
3. Make a table named `my_labels` with the same three columns as `practice_labels`: `complaint_id`, `asks_for_money` and `evidence`. Copy each evidence phrase exactly from the narrative.
4. Show `my_labels` next to each complaint's `product`, `split` and `company_response`. A **merge** joins two tables. `on="complaint_id"` matches each row of `my_labels` to the complaint with the same ID and attaches its fields:

```python
metadata = complaints[["complaint_id", "product", "split", "company_response"]]
my_labels.merge(metadata, on="complaint_id")
```

In [ ]:
# Exercise 3: your code here
# TODO: read five train complaints that are not the three examples in 2.10.
# TODO: make my_labels like practice_labels: complaint_id, asks_for_money and evidence.
# TODO: then attach the other fields by complaint_id (remove the # signs to run these two lines):
# metadata = complaints[["complaint_id", "product", "split", "company_response"]]
# my_labels.merge(metadata, on="complaint_id")

### Practice interpretation

Write 3–5 sentences, using your own outputs:

- The product counts in 2.3 were set by the extract's design. Name one claim they cannot support.
- Which product had the highest monetary relief share in Exercise 1? Is `company_response` a label, a rating or an outcome, and why does that matter for a model that predicts it?
- What do the short narratives (Exercise 2) mean for labeling? Why does it matter for later testing that near-repeat letters stay in one split (2.8)?
- Which of your five labels was hardest to decide, and what change to the policy wording would help?

**Your answer:**

### Submit

1. Run the whole notebook from the top (Restart, then Run All) and check that there are no errors.
2. Save the notebook, so the outputs are saved in the file. The saved outputs are what is graded.
3. In GitHub Desktop, write a short summary such as "Lab 2 complete", click **Commit to main**, then **Push origin**.
4. Push before the deadline. You upload nothing to Canvas: the instructor grades the saved notebook in your repository as it is at the deadline.

If you used an AI tool to explain or debug code, add one line here: the tool, what you used it for, and how you checked the result. Do not paste the 24 messages, your labels or the complaints into an outside tool to get answers.

**AI use:**

## Part 3 Challenge (optional, not graded)

### A. Check labels and evidence for typos

This lists any worksheet row whose label is not one of the three allowed values (for example `Review` with a capital R), or whose evidence is empty. When your worksheet is finished, expect an empty table.

In [ ]:
allowed = ["review", "no_review", "unclear"]
# isin(allowed) is True for an allowed label; ~ turns it into "not allowed".
bad_label = ~labels["label"].isin(allowed)
no_evidence = labels["evidence"].str.strip().eq("")

# Rows with a bad label or (|) empty evidence.
labels.loc[bad_label | no_evidence]

### B. Check that each evidence phrase is really in the message

Evidence should be copied from the message, not paraphrased. This compares each phrase with its message in lower case. Rows with empty evidence count as not found.

In [ ]:
# Check each worksheet row: is its evidence phrase inside its message?
found = []
for text, evidence in zip(labels["text"], labels["evidence"]):
    phrase = evidence.strip()
    # Empty evidence counts as not found. Compare in lower case.
    found.append(phrase != "" and phrase.lower() in text.lower())

labels["evidence_found"] = found
labels[["label", "evidence", "evidence_found"]]

### C. Check the group boundary

This counts, for each group in the partition file, how many partitions it uses. Expect one line: all 23 groups use 1 partition. A 2 would mean an edit put FB005 and FB020 in different partitions. The check cannot find links between customers or conversations that the file does not record.

In [ ]:
# For each group, count how many different partitions its messages use.
partitions_per_group = partition.groupby("group_id")["partition"].nunique()

# How many groups use 1 partition, 2 partitions, and so on.
partitions_per_group.value_counts()

### D. How much does the opening length matter?

Step 2.8 used a 100-character opening, like the course rule. This check asks: if every product-and-opening group had to stay together, how many complaints would need to move to another split? Each group is given the split of its first complaint. The `split` column does not change.

With 100 characters, expect 0. Then change `100` to `60` and run again. Shorter openings join more letters. Read a few: are they really the same letter, or just a common first sentence?

In [ ]:
# The same opening rule as in 2.7. Change 100 to 60 to try shorter openings.
lowered = complaints["narrative"].str.lower()
letters_only = lowered.str.replace(r"[^a-z]+", " ", regex=True)
opening_check = letters_only.str.strip().str[:100]

# For every complaint, look up the split of the first complaint in its group.
# transform("first") gives one value per complaint, not one per group.
group_split = complaints.groupby(["product", opening_check])["split"].transform("first")

# ne() is True where the two splits differ.
print("complaints that would change split:", complaints["split"].ne(group_split).sum())

## Further reading

- [CFPB consumer complaint database narratives archive](https://www.consumerfinance.gov/foia-requests/foia-electronic-reading-room/cfpb-consumer-complaint-database-narratives-archive/): the source of the Practice data.
- [Gebru et al., Datasheets for Datasets](https://arxiv.org/abs/1803.09010): a fuller version of the corpus card idea.
- [pandas: `Series.duplicated`](https://pandas.pydata.org/docs/reference/api/pandas.Series.duplicated.html)

Next, Module 3 prepares the text for analysis while keeping the raw text, the IDs and this data plan.